# 06 - Backpropagation

**AI sin humo** - Notas personales para entender deep learning desde cero.

En el notebook anterior vimos las loss functions: funciones que miden qué tan mal le va al modelo. Ahora viene la pregunta obvia: **¿cómo hacemos para que la loss baje?** Necesitamos calcular cómo cada parámetro contribuye al error, para saber en qué dirección ajustarlos. Eso es **backpropagation** (backprop): el algoritmo para calcular gradientes en toda la red, usando la regla de la cadena.

Este es probablemente el concepto más importante de todo deep learning. Sin backprop, no podríamos entrenar redes profundas. Punto.

---

## Contenido

1. [¿Qué es backpropagation?](#que-es)
2. [La regla de la cadena](#regla-cadena)
3. [Gradient vector](#gradient-vector)
4. [Forward pass vs backward pass](#forward-backward)
5. [Backprop en redes profundas](#backprop-deep)
6. [Grafo computacional y autograd](#grafo-autograd)
7. [Múltiples caminos (batches)](#multiples-caminos)
8. [Operaciones no diferenciables](#no-diferenciables)
9. [Resumen](#resumen)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---

<a id='que-es'></a>
## 1. ¿Qué es backpropagation?

Backprop es **la forma de calcular gradientes para actualizar TODOS los parámetros del modelo**.

Ya sabemos que gradient descent nos dice: "para minimizar la loss, movete en la dirección opuesta al gradiente". Pero en una red con miles o millones de parámetros, ¿cómo calculamos el gradiente de la loss respecto a cada uno de ellos?

La respuesta: usamos la **regla de la cadena** del cálculo, aplicada de forma sistemática desde la loss (al final de la red) hasta los primeros parámetros (cerca del input).

### La intuición

Imaginá que tenés una cadena de operaciones:

```
input → [capa 1] → [capa 2] → ... → [capa N] → loss
```

Cada capa tiene parámetros (weights y biases). Queremos saber: **si cambio un poquito un weight de la capa 1, ¿cuánto cambia la loss?**

El cambio se propaga hacia adelante por todas las capas intermedias. Backprop nos permite calcular esa sensibilidad **de atrás para adelante** (de ahí el nombre), reutilizando cálculos que ya hicimos.

Es super poderoso porque podemos optimizar **cualquier función arbitraria** mientras sus operaciones sean diferenciables. La red se puede diseñar para representar lo que queramos.

---

<a id='regla-cadena'></a>
## 2. La regla de la cadena

Empecemos con lo más simple posible. Tenemos un modelo lineal:

```python
y_hat = w * x + b
loss = (y - y_hat)²
```

Queremos calcular `∂loss/∂w` (cómo cambia la loss si movemos w un poquito).

### Descomponer en sub-operaciones

En vez de intentar derivar todo de una, lo descomponemos en pasos chiquitos:

1. $z = w \cdot x$ (multiplicación)
2. $\hat{y} = z + b$ (suma del bias)
3. $e = y - \hat{y}$ (error)
4. $L = e^2$ (loss cuadrática)

Cada paso es una operación simple cuya derivada conocemos:

| Operación | Derivada local |
|:----------|:--------------:|
| $z = w \cdot x$ | $\frac{\partial z}{\partial w} = x$ |
| $\hat{y} = z + b$ | $\frac{\partial \hat{y}}{\partial z} = 1$ |
| $e = y - \hat{y}$ | $\frac{\partial e}{\partial \hat{y}} = -1$ |
| $L = e^2$ | $\frac{\partial L}{\partial e} = 2e$ |

### Aplicar la regla de la cadena

La regla de la cadena dice: para obtener la derivada total, **multiplicá todas las derivadas locales** a lo largo del camino:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial e} \cdot \frac{\partial e}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} \cdot \frac{\partial z}{\partial w}$$

Reemplazando:

$$\frac{\partial L}{\partial w} = (2e) \cdot (-1) \cdot (1) \cdot (x) = -2(y - \hat{y}) \cdot x$$

Y para el bias:

$$\frac{\partial L}{\partial b} = \frac{\partial L}{\partial e} \cdot \frac{\partial e}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial b} = (2e) \cdot (-1) \cdot (1) = -2(y - \hat{y})$$

La clave es que **cada derivada local es trivial de calcular**. La regla de la cadena nos permite combinarlas para obtener la derivada de cualquier cosa respecto a cualquier cosa.

In [ ]:
# Let's verify the chain rule manually

# Data
x = 2.0
y = 7.0    # target
w = 1.5    # initial weight
b = 0.5    # initial bias

# --- Forward pass: step by step ---
z = w * x           # step 1: z = w * x
y_hat = z + b       # step 2: y_hat = z + b
e = y - y_hat       # step 3: error
loss = e ** 2       # step 4: squared loss

print("=== Forward pass ===")
print(f"z = w * x = {w} * {x} = {z}")
print(f"y_hat = z + b = {z} + {b} = {y_hat}")
print(f"e = y - y_hat = {y} - {y_hat} = {e}")
print(f"loss = e² = {e}² = {loss}")

# --- Backward pass: chain rule step by step ---
dloss_de = 2 * e          # ∂L/∂e = 2e
de_dyhat = -1              # ∂e/∂ŷ = -1
dyhat_dz = 1               # ∂ŷ/∂z = 1
dz_dw = x                  # ∂z/∂w = x
dyhat_db = 1               # ∂ŷ/∂b = 1

# Chain rule: multiply all local derivatives
dloss_dw = dloss_de * de_dyhat * dyhat_dz * dz_dw
dloss_db = dloss_de * de_dyhat * dyhat_db

print("\n=== Backward pass (chain rule) ===")
print(f"∂L/∂e = 2e = 2 * {e} = {dloss_de}")
print(f"∂e/∂ŷ = {de_dyhat}")
print(f"∂ŷ/∂z = {dyhat_dz}")
print(f"∂z/∂w = x = {dz_dw}")
print(f"\n∂L/∂w = {dloss_de} × {de_dyhat} × {dyhat_dz} × {dz_dw} = {dloss_dw}")
print(f"∂L/∂b = {dloss_de} × {de_dyhat} × {dyhat_db} = {dloss_db}")

# --- Verify with numerical gradient (finite differences) ---
eps = 1e-5
loss_w_plus = (y - ((w + eps) * x + b)) ** 2
loss_w_minus = (y - ((w - eps) * x + b)) ** 2
numerical_grad_w = (loss_w_plus - loss_w_minus) / (2 * eps)

print(f"\n=== Verificación numérica ===")
print(f"Gradiente analítico ∂L/∂w = {dloss_dw}")
print(f"Gradiente numérico  ∂L/∂w ≈ {numerical_grad_w:.6f}")
print(f"¡Coinciden! (diferencia: {abs(dloss_dw - numerical_grad_w):.2e})")

Lo que acabamos de hacer es exactamente lo que hace backpropagation, pero a mano y para una función super simple.

Observá algo importante: **cada derivada local es trivial**. No hay nada difícil en calcular que la derivada de $e^2$ es $2e$. La potencia de backprop está en que combina muchas derivadas simples para calcular gradientes en redes arbitrariamente complejas.

---

<a id='gradient-vector'></a>
## 3. Gradient vector

Cuando tenemos más de un parámetro (que es siempre en la práctica), tomamos las derivadas parciales de la loss respecto a **cada uno** de los parámetros. Y de ahí se forma un vector que es el **gradient vector**:

$$\nabla L = \left( \frac{\partial L}{\partial w}, \frac{\partial L}{\partial b} \right)$$

¿Qué nos dice este vector?

- **Dirección**: apunta hacia donde la función **más crece**
- **Magnitud**: nos dice qué tan rápido cambia la función en esa dirección

### La intuición geométrica

Imaginá que tenés una función de 2 variables. Desde un punto, mirás para cada variable cómo cambia la función (manteniendo la otra fija):

- Si aumentar $w$ un poquito hace **disminuir** la loss mucho → la derivada parcial respecto a $w$ es **negativa y grande**
- Si aumentar $b$ un poquito hace **aumentar** la loss muy poquito → la derivada parcial respecto a $b$ es **positiva pero chica**

Ejemplo: si $\frac{\partial L}{\partial w} = -0.1$ y $\frac{\partial L}{\partial b} = 0.02$, el gradient vector es $[-0.1, 0.02]$.

Este vector es una flecha que apunta hacia donde la función **más crece**.

**Entonces, al tomar el negativo del gradient vector, apuntamos al lugar que disminuye la función.** Y eso es exactamente lo que hacemos en gradient descent:

$$\theta \leftarrow \theta - \eta \cdot \nabla L$$

![Gradient vector: apunta hacia donde más crece la función](../ai_notas/AI%20notas/image%2046.png)

In [ ]:
# Visualize the gradient vector on a loss surface

def loss_surface(w, b, x=2.0, y=7.0):
    """MSE loss for y_hat = w*x + b."""
    return (y - (w * x + b)) ** 2

def loss_gradient(w, b, x=2.0, y=7.0):
    """Analytical gradient [∂L/∂w, ∂L/∂b]."""
    y_hat = w * x + b
    e = y - y_hat
    dw = -2 * e * x
    db = -2 * e
    return np.array([dw, db])


w_range = np.linspace(-1, 5, 100)
b_range = np.linspace(-4, 4, 100)
W, B = np.meshgrid(w_range, b_range)
L = np.vectorize(loss_surface)(W, B)

fig, ax = plt.subplots(figsize=(9, 7))
contour = ax.contour(W, B, L, levels=20, cmap='viridis')
ax.clabel(contour, inline=True, fontsize=8)

# Current point and its gradient
w_curr, b_curr = 1.0, -1.0
grad = loss_gradient(w_curr, b_curr)
grad_norm = grad / np.linalg.norm(grad) * 0.8  # normalize for display

# Plot gradient vector (red, points uphill)
ax.annotate('', xy=(w_curr + grad_norm[0], b_curr + grad_norm[1]),
            xytext=(w_curr, b_curr),
            arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
ax.annotate('∇L (crece)', xy=(w_curr + grad_norm[0] + 0.05, b_curr + grad_norm[1] + 0.1),
            fontsize=11, color='red', fontweight='bold')

# Plot negative gradient (blue, points downhill)
ax.annotate('', xy=(w_curr - grad_norm[0], b_curr - grad_norm[1]),
            xytext=(w_curr, b_curr),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2.5))
ax.annotate('-∇L (decrece)', xy=(w_curr - grad_norm[0] + 0.05, b_curr - grad_norm[1] - 0.3),
            fontsize=11, color='blue', fontweight='bold')

ax.plot(w_curr, b_curr, 'ko', markersize=10, label='Punto actual')
ax.plot(3.5, 0.0, 'r*', markersize=15, label='Mínimo (w=3.5, b=0)')

ax.set_xlabel('w', fontsize=12)
ax.set_ylabel('b', fontsize=12)
ax.set_title('Gradient vector sobre la superficie de loss', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

El vector rojo apunta hacia donde la loss **crece más rápido**. El azul (el negativo) apunta hacia donde **decrece más**. Gradient descent simplemente sigue la flecha azul, dando pasitos de tamaño $\eta$ (el learning rate).

En la práctica, un modelo tiene miles o millones de parámetros, entonces el gradient vector vive en un espacio de muchas dimensiones. Pero la idea es exactamente la misma: cada componente del vector te dice cuánto y para dónde mover cada parámetro.

---

<a id='forward-backward'></a>
## 4. Forward pass vs backward pass

El entrenamiento de una red neuronal tiene dos fases clarísimas:

### Forward pass (pasada hacia adelante)

Calculás la predicción del modelo, paso a paso, desde el input hasta la loss. **Y guardás todos los valores intermedios** (activaciones, pre-activaciones, etc.).

```
x → z₁ = w₁x + b₁ → a₁ = σ(z₁) → z₂ = w₂a₁ + b₂ → ŷ = σ(z₂) → e = y - ŷ → L = e²
      ↑ guardar          ↑ guardar       ↑ guardar         ↑ guardar
```

### Backward pass (pasada hacia atrás)

Empezás desde la loss y vas hacia atrás, calculando la derivada local en cada nodo y multiplicándola por el gradiente que viene de arriba (regla de la cadena). **No recalculás valores**: usás los que guardaste en el forward.

```
∂L/∂e → ∂L/∂ŷ → ∂L/∂z₂ → ∂L/∂w₂, ∂L/∂b₂, ∂L/∂a₁ → ∂L/∂z₁ → ∂L/∂w₁, ∂L/∂b₁
```

No estás recalculando derivadas como en symbolic math, ni estás haciendo aproximaciones como en finite differences. Solo:

1. **Forward pass:** calculás y guardás valores intermedios.
2. **Backward pass:** aplicás derivadas locales usando la cadena.

Esto es lo que hace que backprop sea tan eficiente: cada derivada parcial se calcula **una sola vez** y se reutiliza para todos los parámetros que la necesitan.

In [ ]:
# Demonstrate forward and backward pass on a simple 2-layer network

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(a):
    """Derivative of sigmoid, given the output a = sigmoid(z)."""
    return a * (1 - a)


# Data
x = 1.5
y = 0.8  # target

# Parameters (initialized randomly)
np.random.seed(42)
w1, b1 = 0.5, 0.1
w2, b2 = -0.3, 0.7

# === FORWARD PASS ===
# Layer 1
z1 = w1 * x + b1
a1 = sigmoid(z1)

# Layer 2
z2 = w2 * a1 + b2
y_hat = sigmoid(z2)

# Loss
e = y - y_hat
loss = e ** 2

print("=" * 50)
print("FORWARD PASS (calcular y guardar)")
print("=" * 50)
print(f"Layer 1: z1 = {w1}*{x} + {b1} = {z1:.4f}")
print(f"         a1 = σ(z1) = {a1:.4f}")
print(f"Layer 2: z2 = {w2}*{a1:.4f} + {b2} = {z2:.4f}")
print(f"         ŷ  = σ(z2) = {y_hat:.4f}")
print(f"Error:   e  = {y} - {y_hat:.4f} = {e:.4f}")
print(f"Loss:    L  = e² = {loss:.4f}")

# === BACKWARD PASS ===
# Start from the loss and work backwards
dL_de = 2 * e                           # ∂L/∂e
de_dyhat = -1                            # ∂e/∂ŷ
dyhat_dz2 = sigmoid_derivative(y_hat)    # ∂ŷ/∂z₂ = ŷ(1-ŷ)
dz2_dw2 = a1                             # ∂z₂/∂w₂ = a₁  (stored from forward!)
dz2_db2 = 1                              # ∂z₂/∂b₂ = 1
dz2_da1 = w2                             # ∂z₂/∂a₁ = w₂  (stored from forward!)
da1_dz1 = sigmoid_derivative(a1)         # ∂a₁/∂z₁ = a₁(1-a₁)
dz1_dw1 = x                              # ∂z₁/∂w₁ = x   (stored from forward!)
dz1_db1 = 1                              # ∂z₁/∂b₁ = 1

# Chain rule - accumulate from loss to each parameter
# For w2 and b2 (last layer):
dL_dw2 = dL_de * de_dyhat * dyhat_dz2 * dz2_dw2
dL_db2 = dL_de * de_dyhat * dyhat_dz2 * dz2_db2

# For w1 and b1 (first layer - longer chain!):
dL_dw1 = dL_de * de_dyhat * dyhat_dz2 * dz2_da1 * da1_dz1 * dz1_dw1
dL_db1 = dL_de * de_dyhat * dyhat_dz2 * dz2_da1 * da1_dz1 * dz1_db1

print(f"\n{'=' * 50}")
print("BACKWARD PASS (derivadas locales × regla de la cadena)")
print("=" * 50)
print(f"∂L/∂e     = 2e = {dL_de:.4f}")
print(f"∂e/∂ŷ     = {de_dyhat}")
print(f"∂ŷ/∂z₂    = ŷ(1-ŷ) = {dyhat_dz2:.4f}")
print(f"∂z₂/∂w₂   = a₁ = {dz2_dw2:.4f}  ← del forward")
print(f"∂z₂/∂a₁   = w₂ = {dz2_da1}  ← del forward")
print(f"∂a₁/∂z₁   = a₁(1-a₁) = {da1_dz1:.4f}")
print(f"∂z₁/∂w₁   = x = {dz1_dw1}  ← del forward")

print(f"\n--- Gradientes finales ---")
print(f"∂L/∂w₂ = {dL_dw2:.6f}")
print(f"∂L/∂b₂ = {dL_db2:.6f}")
print(f"∂L/∂w₁ = {dL_dw1:.6f}")
print(f"∂L/∂b₁ = {dL_db1:.6f}")

Fijate que para calcular `∂L/∂w1`, necesitamos los mismos primeros tres términos que ya calculamos para `∂L/∂w2`. No los recalculamos: los reutilizamos y simplemente extendemos la cadena con los términos extra. Esta es la razón por la que backprop es eficiente.

---

<a id='backprop-deep'></a>
## 5. Backprop en redes profundas

Veamos esto más formalmente. Tenemos una red simple de 2 capas:

$$z_1 = w_1 x + b_1$$
$$a_1 = \sigma(z_1)$$
$$z_2 = w_2 a_1 + b_2$$
$$\hat{y} = \sigma(z_2)$$
$$e = y - \hat{y}$$
$$L = e^2$$

### Gradiente para w₂ (última capa)

$$\frac{\partial L}{\partial w_2} = \frac{\partial L}{\partial e} \cdot \frac{\partial e}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z_2} \cdot \frac{\partial z_2}{\partial w_2}$$

Cada derivada parcial es fácil de calcular:

$$L = e^2 \Rightarrow \frac{\partial L}{\partial e} = 2e$$

$$e = y - \hat{y} \Rightarrow \frac{\partial e}{\partial \hat{y}} = -1$$

$$\hat{y} = \sigma(z_2) \Rightarrow \frac{\partial \hat{y}}{\partial z_2} = \hat{y}(1-\hat{y})$$

$$z_2 = w_2 a_1 + b_2 \Rightarrow \frac{\partial z_2}{\partial w_2} = a_1$$

Juntando todo:

$$\frac{\partial L}{\partial w_2} = 2e \cdot (-1) \cdot \hat{y}(1-\hat{y}) \cdot a_1$$

### Gradiente para w₁ (primera capa)

$$\frac{\partial L}{\partial w_1} = \underbrace{\frac{\partial L}{\partial e} \cdot \frac{\partial e}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z_2}}_{\text{ya calculado para } w_2} \cdot \frac{\partial z_2}{\partial a_1} \cdot \frac{\partial a_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial w_1}$$

¡Mirá eso! Los primeros tres términos **son exactamente los mismos** que ya calculamos para $w_2$. Solo necesitamos agregar tres derivadas más:

$$\frac{\partial z_2}{\partial a_1} = w_2 \qquad \frac{\partial a_1}{\partial z_1} = a_1(1-a_1) \qquad \frac{\partial z_1}{\partial w_1} = x$$

### Lo que se reutiliza

Como usamos la regla de la cadena, **ya tenemos guardadas casi todas las derivadas parciales** que necesitamos. Solo calculamos las que faltan. Y todos los valores intermedios ($a_1$, $\hat{y}$, $e$, etc.) los guardamos durante el forward pass.

Esto escala a cualquier cantidad de capas. Si tenés 100 capas, la cadena para la capa 1 tiene 100 términos, pero 99 de ellos ya los calculaste cuando procesaste las capas 2 a 100.

> **Nota importante sobre los weights**: En PyTorch, el backward **no modifica** los pesos. Primero se calcula todo el backward (acumulando gradientes), y recién **después** el optimizer actualiza los pesos. Así no hay conflictos.

---

<a id='grafo-autograd'></a>
## 6. Grafo computacional y autograd

En la práctica, nadie calcula gradientes a mano. PyTorch (y otros frameworks) crean un **grafo computacional** automáticamente durante el forward pass.

### ¿Cómo funciona?

Cada operación en la red genera un nodo en el grafo. Cada nodo guarda:

- **`.data`**: el valor numérico real
- **`.grad`**: el gradiente acumulado (se va sumando a medida que llegan gradientes)
- **`._backward`**: la función para calcular el backward desde ese nodo
- **`._prev`**: los nodos anteriores (inputs de esa operación)
- **`._op`**: qué operación generó este nodo

### Paso a paso

**Durante el forward:**
- Cada operación (suma, multiplicación, activación) crea un nuevo nodo
- El nodo guarda su valor y los nodos que lo generaron
- Se forma el grafo automáticamente

**Durante el backward:**
- Se recorre el grafo en **orden topológico inverso** (desde la loss hacia los inputs)
- En cada nodo, se calcula la derivada local y se acumula el gradiente
- **No se recalculan valores intermedios**: usa los guardados del forward

Vamos a implementar una versión simplificada inspirada en [micrograd](https://github.com/karpathy/micrograd) de Karpathy para entender cómo funciona por dentro.

In [ ]:
class Value:
    """A scalar value that tracks operations for automatic differentiation.
    
    Inspired by Karpathy's micrograd. Each Value is a node in the
    computational graph.
    """

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0                     # accumulated gradient
        self._backward = lambda: None       # backward function (default: no-op)
        self._prev = set(_children)         # parent nodes in the graph
        self._op = _op                      # operation that created this node
        self.label = label                  # for visualization

    def __repr__(self):
        return f"Value({self.label}={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            # d(a+b)/da = 1, d(a+b)/db = 1
            # Multiply by upstream gradient (out.grad) — chain rule!
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            # d(a*b)/da = b, d(a*b)/db = a
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f'**{other}')

        def _backward():
            # d(a^n)/da = n * a^(n-1)
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rmul__(self, other):
        return self * other

    def __radd__(self, other):
        return self + other

    def sigmoid(self):
        s = 1 / (1 + np.exp(-self.data))
        out = Value(s, (self,), 'σ')

        def _backward():
            # d(sigmoid(x))/dx = sigmoid(x) * (1 - sigmoid(x))
            self.grad += (s * (1 - s)) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        """Run backpropagation from this node through the entire graph.
        
        Uses reverse topological order so that when we process a node,
        all nodes that depend on it have already propagated their gradients.
        """
        # Build topological order
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # The gradient of the loss with respect to itself is 1
        self.grad = 1.0

        # Walk backwards through the graph
        for node in reversed(topo):
            node._backward()


print("Value class definida. Veamos cómo funciona:")

In [ ]:
# === Demo: simple linear model with our Value class ===

# Create input and parameters as Value nodes
x = Value(2.0, label='x')
y = Value(7.0, label='y')
w = Value(1.5, label='w')
b = Value(0.5, label='b')

# Forward pass — this builds the computational graph automatically!
z = w * x            # z = w * x
z.label = 'z'
y_hat = z + b        # y_hat = z + b
y_hat.label = 'ŷ'
e = y - y_hat        # error
e.label = 'e'
loss = e ** 2        # squared loss
loss.label = 'L'

print("Forward pass completado:")
print(f"  z = w*x = {z.data:.4f}")
print(f"  ŷ = z+b = {y_hat.data:.4f}")
print(f"  e = y-ŷ = {e.data:.4f}")
print(f"  L = e²  = {loss.data:.4f}")

# Backward pass — propagate gradients!
loss.backward()

print(f"\nBackward pass completado:")
print(f"  ∂L/∂w = {w.grad:.4f}  (gradiente de w)")
print(f"  ∂L/∂b = {b.grad:.4f}  (gradiente de b)")
print(f"  ∂L/∂x = {x.grad:.4f}  (gradiente de x, no lo usamos pero se calcula)")

# Verify: analytical gradient is -2 * (y - y_hat) * x = -2 * 3.5 * 2 = -14
print(f"\nVerificación analítica:")
print(f"  ∂L/∂w = -2(y-ŷ)·x = -2·{e.data}·{x.data} = {-2 * e.data * x.data:.4f} ✓")
print(f"  ∂L/∂b = -2(y-ŷ)   = -2·{e.data} = {-2 * e.data:.4f} ✓")

In [ ]:
# === Demo: 2-layer neural network with our Value class ===

# Fresh Values (gradients start at 0)
x = Value(1.5, label='x')
y = Value(0.8, label='y')

# Layer 1 parameters
w1 = Value(0.5, label='w1')
b1 = Value(0.1, label='b1')

# Layer 2 parameters
w2 = Value(-0.3, label='w2')
b2 = Value(0.7, label='b2')

# Forward pass (builds the graph)
z1 = w1 * x + b1
z1.label = 'z1'
a1 = z1.sigmoid()
a1.label = 'a1'

z2 = w2 * a1 + b2
z2.label = 'z2'
y_hat = z2.sigmoid()
y_hat.label = 'ŷ'

e = y - y_hat
e.label = 'e'
loss = e ** 2
loss.label = 'L'

# Backward pass
loss.backward()

print("=== Red de 2 capas con autograd ===")
print(f"\nForward:")
print(f"  z1 = w1*x + b1 = {z1.data:.4f}")
print(f"  a1 = σ(z1) = {a1.data:.4f}")
print(f"  z2 = w2*a1 + b2 = {z2.data:.4f}")
print(f"  ŷ  = σ(z2) = {y_hat.data:.4f}")
print(f"  L  = (y-ŷ)² = {loss.data:.4f}")

print(f"\nGradients (calculados automáticamente):")
print(f"  ∂L/∂w2 = {w2.grad:.6f}")
print(f"  ∂L/∂b2 = {b2.grad:.6f}")
print(f"  ∂L/∂w1 = {w1.grad:.6f}")
print(f"  ∂L/∂b1 = {b1.grad:.6f}")

print(f"\n¡Los gradientes se calcularon automáticamente recorriendo el grafo!")
print(f"Cada nodo solo necesitó saber su derivada local y multiplicar por out.grad.")

In [ ]:
# === Visualize the computational graph ===

def draw_graph(root):
    """Draw the computational graph showing values and gradients."""
    nodes = []
    edges = []
    visited = set()

    def trace(v, depth=0):
        if v not in visited:
            visited.add(v)
            nodes.append((v, depth))
            for child in v._prev:
                edges.append((child, v))
                trace(child, depth + 1)
    trace(root)

    # Simple text visualization
    print("=" * 60)
    print("COMPUTATIONAL GRAPH")
    print("=" * 60)

    # Sort by depth (leaves first)
    nodes.sort(key=lambda x: -x[1])

    for node, depth in nodes:
        indent = "  " * depth
        op_str = f" [{node._op}]" if node._op else " [input]"
        label_str = f"{node.label}: " if node.label else ""
        print(f"{indent}{label_str}data={node.data:.4f}  grad={node.grad:.4f}{op_str}")

draw_graph(loss)

### Entendiendo el código

Mirá lo que pasa en cada operación, por ejemplo la multiplicación:

```python
def __mul__(self, other):
    out = Value(self.data * other.data, (self, other), '*')
    
    def _backward():
        self.grad += other.data * out.grad   # ∂(a*b)/∂a = b
        other.grad += self.data * out.grad   # ∂(a*b)/∂b = a
    out._backward = _backward
    return out
```

Tres cosas clave:

1. **Cada operación solo sabe su derivada local** (para `a*b`: la derivada respecto a `a` es `b`, y viceversa)
2. **Multiplica por `out.grad`** — esto es la regla de la cadena. `out.grad` es el gradiente que viene "de arriba" (desde la loss)
3. **Usa `+=`** — los gradientes se **suman**, no se reemplazan. Esto es crucial cuando un nodo afecta la loss por múltiples caminos (lo veremos en la siguiente sección)

El método `backward()` construye un **orden topológico** del grafo y lo recorre al revés. Así, cuando procesamos un nodo, todos los nodos que dependen de él ya propagaron sus gradientes. Es elegante y eficiente.

In [ ]:
# === Full training loop with our Value class ===
# Let's actually train a tiny network to fit a target!

# Training data: we want to learn f(1.5) ≈ 0.8
x_data = 1.5
y_data = 0.8

# Initialize parameters
w1_val, b1_val = 0.5, 0.1
w2_val, b2_val = -0.3, 0.7
lr = 0.5  # learning rate

losses = []

for step in range(50):
    # Create fresh Value objects each iteration (resets gradients)
    x = Value(x_data)
    y = Value(y_data)
    w1 = Value(w1_val)
    b1 = Value(b1_val)
    w2 = Value(w2_val)
    b2 = Value(b2_val)

    # Forward
    z1 = w1 * x + b1
    a1 = z1.sigmoid()
    z2 = w2 * a1 + b2
    y_hat = z2.sigmoid()
    loss = (y - y_hat) ** 2

    # Backward
    loss.backward()

    # Gradient descent update
    w1_val -= lr * w1.grad
    b1_val -= lr * b1.grad
    w2_val -= lr * w2.grad
    b2_val -= lr * b2.grad

    losses.append(loss.data)

    if step % 10 == 0 or step == 49:
        print(f"Step {step:3d} | loss={loss.data:.6f} | ŷ={y_hat.data:.4f} (target={y_data})")

# Plot the training loss
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses, 'b-', lw=2)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Entrenamiento con nuestro propio autograd')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n¡La loss bajó de {losses[0]:.6f} a {losses[-1]:.6f}!")
print(f"Backprop calculó los gradientes automáticamente en cada paso.")

---

<a id='multiples-caminos'></a>
## 7. Múltiples caminos (batches)

Hasta ahora trabajamos con un solo ejemplo. Pero en la práctica, entrenamos con **batches** de datos. Cuando eso pasa, un mismo parámetro puede afectar la loss por **múltiples caminos** (uno por cada ejemplo del batch).

Por ejemplo, si tenemos un weight $w$ y un batch de 2 ejemplos:

```
          ┌─ x₁ ─→ ŷ₁ ─→ L₁ ─┐
    w ────┤                     ├──→ L_total
          └─ x₂ ─→ ŷ₂ ─→ L₂ ─┘
```

El weight $w$ influye en la loss a través de $x_1$ **y** a través de $x_2$. Entonces la derivada total tiene que **sumar los efectos de todos los caminos**:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial f_1} \frac{\partial f_1}{\partial w} + \frac{\partial L}{\partial f_2} \frac{\partial f_2}{\partial w}$$

### Los gradientes se SUMAN

En el código, esto se manifiesta con `+=`:

```python
# cuando el backward llega al camino 1
w.grad += ∂f1/∂w * grad_output

# cuando el backward llega al camino 2
w.grad += ∂f2/∂w * grad_output
```

PyTorch **siempre** usa `+=`. Si usara `=`, se quedaría con el gradiente del último camino y perdería los demás.

### ¿Por qué `optimizer.zero_grad()`?

Justamente por esto. Como los gradientes se **acumulan** con `+=`, si no los reseteás antes de cada iteración de entrenamiento, los gradientes de la iteración anterior se **suman** a los de la nueva. Eso arruina todo.

```python
# El loop de entrenamiento típico en PyTorch:
for batch in dataloader:
    optimizer.zero_grad()     # ← RESETEAR gradientes (fundamental)
    output = model(batch)     # forward
    loss = criterion(output)  # calcular loss
    loss.backward()           # backward (acumula gradientes)
    optimizer.step()          # actualizar parámetros
```

Si te olvidás del `zero_grad()`, el modelo no va a converger bien. Es uno de los bugs más comunes al empezar con PyTorch.

In [ ]:
# Demo: gradient accumulation with multiple paths

# A weight 'w' affects the loss through 2 different data points
w = Value(2.0, label='w')

# Path 1: data point (x1=1, y1=3)
x1 = Value(1.0, label='x1')
y1 = Value(3.0, label='y1')
pred1 = w * x1
loss1 = (y1 - pred1) ** 2

# Path 2: data point (x2=2, y2=5)
x2 = Value(2.0, label='x2')
y2 = Value(5.0, label='y2')
pred2 = w * x2
loss2 = (y2 - pred2) ** 2

# Total loss = loss1 + loss2
total_loss = loss1 + loss2

# Backward: gradients accumulate via +=
total_loss.backward()

print("Múltiples caminos: w afecta la loss por 2 data points")
print(f"\n  w = {w.data}")
print(f"  Path 1: pred1 = w*x1 = {w.data}*{x1.data} = {pred1.data}, loss1 = {loss1.data}")
print(f"  Path 2: pred2 = w*x2 = {w.data}*{x2.data} = {pred2.data}, loss2 = {loss2.data}")
print(f"  Total loss = {total_loss.data}")
print(f"\n  ∂L/∂w = {w.grad:.4f}  ← sum of contributions from both paths")

# Manual verification
# ∂loss1/∂w = -2 * (y1 - w*x1) * x1 = -2 * (3 - 2) * 1 = -2
# ∂loss2/∂w = -2 * (y2 - w*x2) * x2 = -2 * (5 - 4) * 2 = -4
# Total: -2 + (-4) = -6
manual_grad1 = -2 * (3.0 - 2.0 * 1.0) * 1.0
manual_grad2 = -2 * (5.0 - 2.0 * 2.0) * 2.0
print(f"\nVerificación manual:")
print(f"  ∂L₁/∂w = {manual_grad1:.4f}  (contribución del path 1)")
print(f"  ∂L₂/∂w = {manual_grad2:.4f}  (contribución del path 2)")
print(f"  Total   = {manual_grad1 + manual_grad2:.4f}  ← se SUMAN ✓")

In [ ]:
# What happens if we DON'T zero gradients between iterations?

w = Value(1.0, label='w')
x = Value(2.0, label='x')
y = Value(5.0, label='y')

print("BUG DEMO: qué pasa si NO hacemos zero_grad()\n")

for i in range(3):
    # ⚠️ NOT resetting w.grad!
    pred = w * x
    loss = (y - pred) ** 2
    loss.backward()

    print(f"  Iter {i}: loss={loss.data:.2f}, w.grad={w.grad:.2f} "
          f"{'← correcto' if i == 0 else '← ¡ACUMULADO! grad de iteraciones anteriores'}")

print(f"\n  El gradiente debería ser el mismo en cada iteración (mismos datos),")
print(f"  pero se acumula porque no reseteamos.")
print(f"  En PyTorch: optimizer.zero_grad() al inicio de cada iteración.")

---

<a id='no-diferenciables'></a>
## 8. Operaciones no diferenciables

Una pregunta que surge naturalmente: **¿toda la red tiene que ser diferenciable?**

La respuesta corta: **no**. Una red neuronal no tiene que ser 100% diferenciable en todas sus partes. Solo tiene que serlo **en el camino entre los pesos aprendibles y la función de pérdida**.

Se pueden tener operaciones no diferenciables si se cumple alguna de estas condiciones:

### 1. No tienen parámetros aprendibles

Si una operación no tiene weights que optimizar (por ejemplo, un `if`, una regla fija, un umbral), no necesita ser diferenciable. No hay nada que optimizar ahí, así que no necesitamos calcular gradientes para ella.

Ejemplos:
- **ReLU** en $x=0$: técnicamente no es diferenciable, pero le asignamos derivada 0 y funciona perfecto
- **Argmax** para seleccionar la clase predicha: no es diferenciable, pero se usa solo en inferencia, no afecta el entrenamiento
- **Rounding/discretization**: si es parte del preprocesamiento (antes de los weights), no importa

### 2. No interrumpen el flujo del gradiente

Si tenés operaciones diferenciables con parámetros aprendibles, y después de ellas ponés una operación no diferenciable, **la señal del gradiente nunca va a llegar a los parámetros aprendibles**. Se corta la cadena.

```
✓ FUNCIONA:  x → [diferenciable con weights] → [no-diferenciable sin weights] → loss
                  (el gradiente llega a los weights a través de la parte diferenciable)

✗ NO FUNCIONA: x → [no-diferenciable] → [diferenciable con weights] → loss
                   (el gradiente se corta antes de llegar a los weights anteriores)
```

La regla de oro: **el gradiente tiene que poder fluir sin interrupciones desde la loss hasta cada parámetro que querés optimizar.** Todo lo demás puede ser lo que quieras.

In [ ]:
# Example: operations in the loss pipeline

# Cross-entropy loss decomposes into differentiable sub-operations:
# predictions → softmax → log → sum → negate → loss

# Each step is differentiable:
logits = np.array([2.0, 1.0, 0.1])
target_class = 0

# Step 1: softmax (differentiable)
exp_logits = np.exp(logits - np.max(logits))
probs = exp_logits / exp_logits.sum()

# Step 2: select target prob (indexing — not differentiable, but no learnable params!)
target_prob = probs[target_class]

# Step 3: log (differentiable)
log_prob = np.log(target_prob)

# Step 4: negate (differentiable, trivially)
loss = -log_prob

print("Cross-entropy loss, paso a paso:")
print(f"  Logits:     {logits}")
print(f"  Softmax:    {np.round(probs, 4)}       ← diferenciable")
print(f"  Select [0]: {target_prob:.4f}             ← no diferenciable, pero sin params")
print(f"  Log:        {log_prob:.4f}             ← diferenciable")
print(f"  Negate:     {loss:.4f}              ← diferenciable")
print(f"\n  El indexing (seleccionar la clase correcta) no es diferenciable,")
print(f"  pero no importa porque no tiene parámetros aprendibles.")
print(f"  El gradiente fluye sin problemas a través de las demás operaciones.")

---

<a id='resumen'></a>
## 9. Resumen

| Concepto | Descripción |
|:---------|:------------|
| **Backpropagation** | Algoritmo para calcular gradientes de la loss respecto a todos los parámetros, usando la regla de la cadena |
| **Regla de la cadena** | Descomponer en sub-operaciones y multiplicar derivadas locales: $\frac{\partial L}{\partial w} = \frac{\partial L}{\partial e} \cdot \frac{\partial e}{\partial \hat{y}} \cdot ...$ |
| **Gradient vector** | Vector de derivadas parciales. Apunta donde la función crece. El negativo apunta donde decrece |
| **Forward pass** | Calcular la predicción y **guardar** todos los valores intermedios |
| **Backward pass** | Recorrer la red al revés, aplicando derivadas locales. Reutiliza valores del forward |
| **Grafo computacional** | Estructura que registra todas las operaciones. Cada nodo tiene `.data`, `.grad`, `._backward`, `._prev` |
| **Autograd** | PyTorch construye el grafo automáticamente y lo recorre en orden topológico inverso |
| **Gradientes se suman** | Cuando un parámetro afecta la loss por múltiples caminos (batches), los gradientes se acumulan con `+=` |
| **`zero_grad()`** | Resetear gradientes antes de cada iteración. Sin esto, se acumulan gradientes de iteraciones anteriores |
| **Operaciones no diferenciables** | Se permiten si: (1) no tienen params aprendibles, o (2) no interrumpen el flujo del gradiente |

### La clave de todo

Backprop es lo que hace posible entrenar redes profundas. Es un algoritmo elegante que:

1. **Descompone** funciones complejas en operaciones simples
2. **Calcula** derivadas locales triviales para cada operación
3. **Combina** todo con la regla de la cadena
4. **Reutiliza** cálculos para ser eficiente

Y lo más potente: podemos optimizar **cualquier función arbitraria** mientras sus operaciones sean diferenciables. La red se puede diseñar para representar lo que queramos.

---

**Siguiente notebook →** [07 - Entrenamiento Práctico](./07_entrenamiento_practico.ipynb): problemas que aparecen en la práctica (inestabilidad, convergencia lenta, mínimos malos) y cómo solucionarlos con optimizers, normalization e initialization.